In [2]:
import pandas as pd
from io import BytesIO, StringIO
from zipfile import ZipFile
import glob
import csv
import numpy as np


In [3]:
path = "/Volumes/2/casa/data/*.csv"
for filename in glob.glob(path):
    reader = pd.read_csv(filename, sep = None, iterator = True, engine = 'python')
    inferred_separator = reader._engine.data.dialect.delimiter
    print(inferred_separator, filename, 'TIMESTAMP' in pd.read_csv(filename, nrows=0, sep=inferred_separator).columns.tolist())

, /Volumes/2/casa/data/salesstock$vw-04-06_2023.csv True
; /Volumes/2/casa/data/salesstock$vw-10-12_2022.csv True
; /Volumes/2/casa/data/salesstock$vw-09_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-10-12_2021.csv True
; /Volumes/2/casa/data/salesstock$vw-03_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-04_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-12_2018.csv True
; /Volumes/2/casa/data/salesstock$vw-12_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-07-08_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-11-12_2020.csv True
, /Volumes/2/casa/data/salesstock$vw-05-06_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2023.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2022.csv True
, /Volumes/2/casa/data/salesstock$vw-04-05_2021.csv True
, /Volumes/2/casa/data/salesstock$vw-04-05_2020.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2021.csv True
; /Volumes/2/casa/data/salesstock$vw-11_2018.csv True
; /Volumes/2/casa/data/salesstock$vw-10_2018.csv 

In [3]:
path1 = "/Volumes/2/casa/timestamp/*.csv"
path2 = "/Volumes/2/casa/data/*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, filename in enumerate(path_list1):
    print(i+1, filename, end='\n')

print(set(path_list1) <= (set(path_list2)))

1 salesstock$vw-04-06_2023.csv
2 salesstock$vw-10-12_2022.csv
3 salesstock$vw-09_2019.csv
4 salesstock$vw-10-12_2021.csv
5 salesstock$vw-03_2019.csv
6 salesstock$vw-04_2019.csv
7 salesstock$vw-12_2018.csv
8 salesstock$vw-12_2019.csv
9 salesstock$vw-07-08_2019.csv
10 salesstock$vw-11-12_2020.csv
11 salesstock$vw-07-09_2023.csv
12 salesstock$vw-07-09_2022.csv
13 salesstock$vw-04-05_2021.csv
14 salesstock$vw-07-09_2021.csv
15 salesstock$vw-11_2018.csv
16 salesstock$vw-10_2018.csv
17 salesstock$vw-10_2019.csv
18 salesstock$vw-11_2019.csv
19 salesstock$vw-06-07_2020.csv
20 salesstock$vw-06_2021.csv
21 salesstock$vw-01-03_2021.csv
22 salesstock$vw-08-09_2020.csv
23 salesstock$vw-01-03_2020.csv
24 salesstock$vw-10_2020.csv
25 salesstock$vw-01-03_2023.csv
True


In [4]:
folder1 = "/Volumes/2/casa/timestamp/"
folder2 = "/Volumes/2/casa/data/"

path1 = folder1 + "*.csv"
path2 = folder2 + "*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, path in enumerate(path_list2):
    reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
    inferred_separator2 = reader2._engine.data.dialect.delimiter
    if 'TIMESTAMP' not in pd.read_csv(folder2 + path, nrows=0, sep=inferred_separator2).columns.tolist():
        print(i, path)

7 salesstock$vw-12_2019.csv
9 salesstock$vw-11-12_2020.csv
12 salesstock$vw-07-09_2022.csv
15 salesstock$vw-07-09_2021.csv
17 salesstock$vw-10_2018.csv
20 salesstock$vw-06-07_2020.csv
21 salesstock$vw-06_2021.csv
23 salesstock$vw-01-03_2021.csv
24 salesstock$vw-08-09_2020.csv
25 salesstock$vw-01-03_2020.csv


In [3]:
folder1 = "/Volumes/2/casa/timestamp/"
folder2 = "/Volumes/2/casa/data/"

path1 = folder1 + "*.csv"
path2 = folder2 + "*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, path in enumerate(path_list2):
    reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
    inferred_separator2 = reader2._engine.data.dialect.delimiter
    if 'TIMESTAMP' not in pd.read_csv(folder2 + path, nrows=0, sep=inferred_separator2).columns.tolist():
        reader1 = pd.read_csv(folder1 + path, sep = None, iterator = True, engine = 'python')
        inferred_separator1 = reader1._engine.data.dialect.delimiter
        try:
            df1 = pd.read_csv(folder1 + path, low_memory=False, sep=inferred_separator1)
            df2 = pd.read_csv(folder2 + path, low_memory=False, sep=inferred_separator2)
        except Exception as e:
            print(e, i, path, end='\n')
            continue
        if df1['SALESSTOCKID'].equals(df2['SALESSTOCKID']):
            df2['TIMESTAMP'] = df1['TIMESTAMP']
            df2.to_csv(folder2 + path, index=False, quoting=csv.QUOTE_ALL, sep=inferred_separator2)
            print("EQ", i, path, end='\n')
        else:
            # df2.merge(df1, on=['SALESSTOCKID'], how='left').to_csv(folder2 + path, index=False)
            print("MERGE", i, path, end='\n')

EQ 8 salesstock$vw-04_2019.csv
EQ 18 salesstock$vw-07-09_2021.csv
EQ 20 salesstock$vw-10_2018.csv
EQ 25 salesstock$vw-08-09_2020.csv
EQ 26 salesstock$vw-01-03_2020.csv


In [4]:
folder_data = "/Volumes/2/casa/data/"
folder_daily = "/Volumes/2/casa/daily_usercount/"
file_list = [filename.split("/")[-1] for filename in glob.glob(folder_daily + "*.csv")]
header = ['TIMESTAMP', 'SECONDSSPENT', 'QUANTITY', 'VOLUME', 'WEIGHT', 'PRICE', 'TEAMNAME', 'USERNAME']

for i, filepath in enumerate(glob.glob(folder_data + "*.csv")):
    if filepath.split("/")[-1] not in file_list:
        try:
            print(filepath)
            reader = pd.read_csv(filepath, sep = None, iterator = True, engine = 'python')
            inferred_separator = reader._engine.data.dialect.delimiter

            df = pd.read_csv(filepath, usecols=header, low_memory=False, sep=inferred_separator)[header]
            df['VOLUME'] = df['VOLUME'].str.replace('[^\d\,]', '', regex=True)
            df['VOLUME'] = df['VOLUME'].str.replace('^\s*', '0', regex=True)
            df['VOLUME'] = df['VOLUME'].str.replace(',', '.').astype(float)
            df['WEIGHT'] = df['WEIGHT'].str.replace('[^\d\,]', '', regex=True)
            df['WEIGHT'] = df['WEIGHT'].str.replace('^\s*', '0', regex=True)
            df['WEIGHT'] = df['WEIGHT'].str.replace(',', '.').astype(float)
            df['PRICE'] = df['PRICE'].str.replace('[^\d\,]', '', regex=True)
            df['PRICE'] = df['PRICE'].str.replace('^\s*', '0', regex=True)
            df['PRICE'] = df['PRICE'].str.replace(',', '.').astype(float)

            # df['TIMESTAMP'] = df['TIMESTAMP'].str.strip()
            df['DATE'] = df['TIMESTAMP'].str.split(" ", expand=True)[0]
            df['DATE'] = df['DATE'].str.zfill(10)
            df['DATE'] = pd.to_datetime(df['DATE'], format="%d/%m/%Y")
            df = df.drop(['TIMESTAMP'], axis=1)
            df = (df.groupby(['DATE', 'TEAMNAME'])
                .agg({'USERNAME':'nunique', 'SECONDSSPENT': 'sum', 'QUANTITY': 'sum', 'VOLUME': 'sum', 'WEIGHT': 'sum', 'PRICE': 'sum'})
                .reset_index()
                .rename(columns={'USERNAME': 'USERCOUNT'})
            )
            df.sort_values(by=['DATE'], inplace=True)
            df.set_index('DATE', inplace=True)
            df.to_csv(folder_daily + filepath.split("/")[-1], index=True, quoting=csv.QUOTE_ALL, sep=inferred_separator)
            print(i+1, filepath.split("/")[-1], end='\n')
        except Exception as e:
            print(i+1, filepath.split("/")[-1], e)
            continue


/Volumes/2/casa/data/salesstock$vw-04-06_2023.csv
1 salesstock$vw-04-06_2023.csv
/Volumes/2/casa/data/salesstock$vw-10-12_2022.csv
2 salesstock$vw-10-12_2022.csv
/Volumes/2/casa/data/salesstock$vw-09_2019.csv
3 salesstock$vw-09_2019.csv
/Volumes/2/casa/data/salesstock$vw-10-12_2021.csv
4 salesstock$vw-10-12_2021.csv
/Volumes/2/casa/data/salesstock$vw-03_2019.csv
5 salesstock$vw-03_2019.csv
/Volumes/2/casa/data/salesstock$vw-04_2019.csv
6 salesstock$vw-04_2019.csv
/Volumes/2/casa/data/salesstock$vw-12_2018.csv
7 salesstock$vw-12_2018.csv
/Volumes/2/casa/data/salesstock$vw-12_2019.csv
8 salesstock$vw-12_2019.csv
/Volumes/2/casa/data/salesstock$vw-07-08_2019.csv
9 salesstock$vw-07-08_2019.csv
/Volumes/2/casa/data/salesstock$vw-11-12_2020.csv
10 salesstock$vw-11-12_2020.csv
/Volumes/2/casa/data/salesstock$vw-05-06_2019.csv
11 salesstock$vw-05-06_2019.csv
/Volumes/2/casa/data/salesstock$vw-07-09_2023.csv
12 salesstock$vw-07-09_2023.csv
/Volumes/2/casa/data/salesstock$vw-07-09_2022.csv
13 sa

In [ ]:
# # CHECK LATER !!! all the files with timestamp column

# path = 'salesstock$vw-04-06_2023.csv'

# reader1 = pd.read_csv(folder1 + path, sep = None, iterator = True, engine = 'python')
# inferred_separator1 = reader1._engine.data.dialect.delimiter
# reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
# inferred_separator2 = reader2._engine.data.dialect.delimiter

# df1 = pd.read_csv(folder1 + path, low_memory=False, sep=inferred_separator1)
# df2 = pd.read_csv(folder2 + path, low_memory=False, sep=inferred_separator2)
# print(len(df1), len(df2))
# # df2.merge(df1, on='SALESSTOCKID', how='left')

8515814 8515814


In [4]:
pd.options.display.max_columns = None
usecols = ['UITERSTE BESTELDT.', 'WOANBL', 'ORDER-TYPE LOCUS']
df_wo = pd.read_csv('/Volumes/2/labour planning/WO.csv', usecols=usecols)
df_wo['DATE'] = df_wo['UITERSTE BESTELDT.']
df_wo['LOCUS'] = df_wo['ORDER-TYPE LOCUS']
df_wo.drop(['UITERSTE BESTELDT.', 'ORDER-TYPE LOCUS'], axis=1, inplace=True)
df_wo['DATE'] = df_wo['DATE'].astype(str).apply(lambda x: x[:4] + '-' + x[4:6] + '-' + x[6:])
df_wo = df_wo.groupby(['DATE', 'LOCUS']).sum().reset_index()
df_wo.to_csv('daily_wo.csv', index=False)
df_wo

,DATE,LOCUS,WOANBL
0,2018-07-01,NL,35241
1,2018-07-02,NL,17837
2,2018-07-02,TL,7
3,2018-07-03,NL,30344
4,2018-07-04,NL,27013
...,...,...,...
4690,2023-10-09,NL,0
4691,2023-10-10,NL,0
4692,2023-10-11,NL,0
4693,2023-10-12,NL,0


In [5]:
pd.options.display.max_columns = None
usecols = ['DATE', 'BESTELD', 'VERKOOP             PRIJS', 'ORDER-TYPE LOCUS']
df_wl = pd.read_csv('/Volumes/2/labour planning/WL.csv', usecols=usecols, low_memory=False)[usecols]
df_wl['LOCUS'] = df_wl['ORDER-TYPE LOCUS']
df_wl['ORDER'] = df_wl['BESTELD']
df_wl['PRICE'] = df_wl['VERKOOP             PRIJS']
df_wl.drop(['BESTELD', 'VERKOOP             PRIJS', 'ORDER-TYPE LOCUS'], axis=1, inplace=True)
df_wl['DATE'] = df_wl['DATE'].astype(str).apply(lambda x: x[:4] + '-' + x[4:6] + '-' + x[6:])
df_wl = df_wl.groupby(['DATE', 'LOCUS']).sum().reset_index()
df_wl.to_csv('daily_wl.csv', index=False)
df_wl


,DATE,LOCUS,ORDER,PRICE
0,2018-07-01,NL,5400,9610.54
1,2018-07-01,NLECO,624,693.94
2,2018-07-02,ARCA,17978,24995.20
3,2018-07-02,ARWEB,653,2584.60
4,2018-07-02,KO,1122,11991.27
...,...,...,...,...
14106,2023-09-07,KO,171,7894.04
14107,2023-09-07,NL,13700,44321.80
14108,2023-09-07,NLECO,3546,1112.55
14109,2023-09-07,TL,348,345.40


In [16]:
df_wl = pd.read_csv('daily_wl.csv')
print(df_wl['LOCUS'].unique())
df_wl.drop(['LOCUS'], axis=1, inplace=True)
df_wl_total = df_wl.groupby(['DATE']).sum().reset_index()
df_wl_total.to_csv('daily_wl_total.csv', index=False)

['NL' 'NLECO' 'ARCA' 'ARWEB' 'KO' 'PHSLD' 'VFOLD' 'VNART' 'VSOLD' 'VWK'
 'TL' 'NWDSP' 'NWECO' 'NWKL' 'EXT' 'NLKER' 'PHKER' 'VPLP' 'VRET' 'VKER'
 'NLPAS' 'VPAS' 'WP' 'WC']


In [21]:
df_wo = pd.read_csv('daily_wo.csv')
print(df_wo['LOCUS'].unique())
df_wo.drop(['LOCUS'], axis=1, inplace=True)
df_wo_total = df_wo.groupby(['DATE']).sum().reset_index()
df_wo_total.to_csv('daily_wo_total.csv', index=False)

['NL' 'TL' 'NWDSP' 'NWECO' 'PHSLD' 'ARWEB' 'NWKL' 'EXT' 'PHKER' 'NLECO'
 'VRET' 'VSOLD' 'VFOLD' 'V2ND' 'NLKER' 'WP' 'WC' 'VKER']
